In [ ]:
!pip -q install -U segmentation-models-pytorch==0.3.3 timm "albumentations>=1.4.0,<1.5.0" opencv-python

In [ ]:
import os, glob, random, time
from typing import List, Tuple, Dict
from dataclasses import dataclass

import numpy as np
import cv2
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

import segmentation_models_pytorch as smp

In [ ]:
import json, pathlib, datetime

RUN_ID = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
OUT_DIR = pathlib.Path("/kaggle/working") / f"runs_{RUN_ID}"
OUT_DIR.mkdir(parents=True, exist_ok=True)

LOG_FILE = OUT_DIR / "train_log.txt"

def log(msg: str):
    print(msg)
    with open(LOG_FILE, "a") as f:
        f.write(msg + "\n")

log(f"[RUN_ID] {RUN_ID}")
log(f"[OUT_DIR] {OUT_DIR}")

In [ ]:
# ---------------------------
# 1) Repro + device
# ---------------------------
SEED = 42
def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
# ============================================================
# 2) Paths
# ============================================================

KVASIR_ROOT = "/kaggle/input/datasets/debeshjha1/kvasirseg/Kvasir-SEG/Kvasir-SEG"
KVASIR_IMAGES_DIR = f"{KVASIR_ROOT}/images"
KVASIR_MASKS_DIR  = f"{KVASIR_ROOT}/masks"

# Optional external dataset (not the highlight, but supported)
USE_EXTERNAL_TEST = True
CVC_ROOT = "/kaggle/input/datasets/balraj98/cvcclinicdb/PNG"
CVC_IMAGES_DIR = os.path.join(CVC_ROOT, "Original")
CVC_MASKS_DIR  = os.path.join(CVC_ROOT, "Ground Truth")

def assert_dir_exists(path: str):
    if not os.path.isdir(path):
        raise FileNotFoundError(f"Directory not found: {path}")

assert_dir_exists(KVASIR_IMAGES_DIR)
assert_dir_exists(KVASIR_MASKS_DIR)

In [ ]:
# ============================================================
# 3) Pair listing
# ============================================================
def list_image_mask_pairs(images_dir: str, masks_dir: str, img_exts=(".jpg",".jpeg",".png")) -> List[Tuple[str,str]]:
    img_files = []
    for e in img_exts:
        img_files.extend(glob.glob(os.path.join(images_dir, f"*{e}")))
    img_files = sorted(img_files)

    mask_files = sorted(glob.glob(os.path.join(masks_dir, "*")))
    mask_map = {os.path.splitext(os.path.basename(m))[0]: m for m in mask_files}

    pairs = []
    for im in img_files:
        stem = os.path.splitext(os.path.basename(im))[0]
        if stem in mask_map:
            pairs.append((im, mask_map[stem]))
    return pairs

kvasir_pairs = list_image_mask_pairs(KVASIR_IMAGES_DIR, KVASIR_MASKS_DIR)
print("Kvasir pairs:", len(kvasir_pairs))
if len(kvasir_pairs) != 1000:
    print("WARNING: Kvasir is typically 1000 pairs. Check paths.")

external_pairs = []
if USE_EXTERNAL_TEST:
    assert_dir_exists(CVC_IMAGES_DIR)
    assert_dir_exists(CVC_MASKS_DIR)
    external_pairs = list_image_mask_pairs(CVC_IMAGES_DIR, CVC_MASKS_DIR, img_exts=(".png",".jpg",".jpeg"))
    print("CVC pairs:", len(external_pairs))

In [ ]:
# ============================================================
# 4) Protocol A split (900 train / 100 test) + internal val
# ============================================================
def split_protocol_A(pairs: List[Tuple[str,str]], seed=SEED) -> Dict[str, List[Tuple[str,str]]]:
    assert len(pairs) >= 1000, "Need 1000 pairs for Protocol A."
    idx = np.arange(len(pairs))
    rng = np.random.default_rng(seed)
    rng.shuffle(idx)

    test_idx = idx[:100]
    trainval_idx = idx[100:1000]  # 900

    trainval = [pairs[i] for i in trainval_idx]
    test = [pairs[i] for i in test_idx]

    # val from trainval
    rng2 = np.random.default_rng(seed + 1)
    tv_idx = np.arange(len(trainval))
    rng2.shuffle(tv_idx)
    val_idx = tv_idx[:100]
    train_idx = tv_idx[100:]

    train = [trainval[i] for i in train_idx]  # 800
    val = [trainval[i] for i in val_idx]      # 100
    return {"train": train, "val": val, "test": test}

splits = split_protocol_A(kvasir_pairs, seed=SEED)
print({k: len(v) for k, v in splits.items()})

In [ ]:
# ============================================================
# 5) Edge GT utility
# ============================================================
def mask_to_edge(mask01: np.ndarray, ksize: int = 3) -> np.ndarray:
    """
    mask01: HxW binary (0/1)
    returns edge map HxW binary (0/1)
    """
    m = (mask01.astype(np.uint8) * 255)
    kernel = np.ones((ksize, ksize), np.uint8)
    dil = cv2.dilate(m, kernel, iterations=1)
    ero = cv2.erode(m, kernel, iterations=1)
    edge = cv2.absdiff(dil, ero)
    edge = (edge > 0).astype(np.float32)
    return edge

In [ ]:
# ============================================================
# 6) Transforms + Dataset (supports patch training)
# ============================================================
@dataclass
class TrainConfig:
    use_patch_training: bool = True
    patch_size: int = 256
    img_size_eval: int = 384  # used for val/test resize

def get_transforms(cfg: TrainConfig):
    add_targets = {"edge": "mask"}  # <-- IMPORTANT

    if cfg.use_patch_training:
        train_tfms = A.Compose([
            A.PadIfNeeded(min_height=cfg.patch_size, min_width=cfg.patch_size,
                          border_mode=cv2.BORDER_CONSTANT, value=0, mask_value=0),
            A.RandomCrop(cfg.patch_size, cfg.patch_size),
            A.HorizontalFlip(p=0.5),
            A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.10, rotate_limit=15,
                               border_mode=cv2.BORDER_CONSTANT, value=0, mask_value=0, p=0.5),
            A.RandomBrightnessContrast(p=0.3),
            A.GaussNoise(p=0.2),
            A.Normalize(),
            ToTensorV2()
        ], additional_targets=add_targets)
    else:
        train_tfms = A.Compose([
            A.Resize(cfg.img_size_eval, cfg.img_size_eval),
            A.HorizontalFlip(p=0.5),
            A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.10, rotate_limit=15,
                               border_mode=cv2.BORDER_CONSTANT, value=0, mask_value=0, p=0.5),
            A.RandomBrightnessContrast(p=0.3),
            A.GaussNoise(p=0.2),
            A.Normalize(),
            ToTensorV2()
        ], additional_targets=add_targets)

    val_tfms = A.Compose([
        A.Resize(cfg.img_size_eval, cfg.img_size_eval),
        A.Normalize(),
        ToTensorV2()
    ], additional_targets=add_targets)

    return train_tfms, val_tfms

class PolypDatasetEdge(Dataset):
    def __init__(self, pairs: List[Tuple[str,str]], transform=None):
        self.pairs = pairs
        self.transform = transform

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, mask_path = self.pairs[idx]

        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = (mask > 127).astype(np.float32)  # 0/1
        edge = mask_to_edge(mask, ksize=3)

        if self.transform:
            aug = self.transform(image=img, mask=mask, edge=edge)
            img = aug["image"]
            mask = aug["mask"]
            edge = aug["edge"]

        # ToTensorV2 converts masks to torch tensors but keeps shape HxW
        mask = mask.unsqueeze(0).float()  # 1xHxW
        edge = edge.unsqueeze(0).float()  # 1xHxW
        return img, mask, edge

In [ ]:
# ============================================================
# 7) Model: UNet++ with Edge Auxiliary Head
# ============================================================
class UNetPPWithEdgeHead(nn.Module):
    """
    UNet++ where we expose decoder output and add an edge head on it.
    Works with smp.UnetPlusPlus that has encoder/decoder/segmentation_head.
    """
    def __init__(self, encoder_name="timm-efficientnet-b7"):
        super().__init__()
        self.net = smp.UnetPlusPlus(
            encoder_name=encoder_name,
            encoder_weights="imagenet",
            in_channels=3,
            classes=1,
            activation=None,
        )
        # segmentation head input channels:
        in_ch = self.net.segmentation_head[0].in_channels if isinstance(self.net.segmentation_head, nn.Sequential) else self.net.segmentation_head.in_channels

        # edge head: lightweight
        self.edge_head = nn.Sequential(
            nn.Conv2d(in_ch, in_ch // 2, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(in_ch // 2),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_ch // 2, 1, kernel_size=1)
        )

    def forward(self, x):
        feats = self.net.encoder(x)
        dec = self.net.decoder(*feats)
        mask_logits = self.net.segmentation_head(dec)
        edge_logits = self.edge_head(dec)
        return mask_logits, edge_logits

In [ ]:
# ============================================================
# 8) Losses + Metrics
# ============================================================
bce = nn.BCEWithLogitsLoss()

def dice_loss_from_logits(logits, targets, eps=1e-7):
    probs = torch.sigmoid(logits)
    probs = probs.view(probs.size(0), -1)
    targets = targets.view(targets.size(0), -1)
    inter = (probs * targets).sum(dim=1)
    union = probs.sum(dim=1) + targets.sum(dim=1)
    dice = (2.0 * inter + eps) / (union + eps)
    return 1.0 - dice.mean()

def seg_loss(mask_logits, mask_gt, w_bce=0.5, w_dice=0.5):
    return w_bce * bce(mask_logits, mask_gt) + w_dice * dice_loss_from_logits(mask_logits, mask_gt)

def edge_loss(edge_logits, edge_gt):
    return bce(edge_logits, edge_gt)

@torch.no_grad()
def dice_iou_from_logits(mask_logits, mask_gt, thr=0.5, eps=1e-7):
    probs = torch.sigmoid(mask_logits)
    pred = (probs > thr).float()
    pred = pred.view(pred.size(0), -1)
    gt = mask_gt.view(mask_gt.size(0), -1)
    inter = (pred * gt).sum(dim=1)
    dice = (2.0 * inter + eps) / (pred.sum(dim=1) + gt.sum(dim=1) + eps)
    iou  = (inter + eps) / (pred.sum(dim=1) + gt.sum(dim=1) - inter + eps)
    return dice.mean().item(), iou.mean().item()

In [ ]:
# ============================================================
# 9) Inference: TTA + Post-processing (Largest Component)
# ============================================================
@torch.no_grad()
def predict_logits_tta(model, x):
    """
    x: (B,3,H,W)
    flips-based TTA
    returns averaged mask logits
    """
    model.eval()
    m0, _ = model(x)

    x_h = torch.flip(x, dims=[3])
    mh, _ = model(x_h)
    mh = torch.flip(mh, dims=[3])

    x_v = torch.flip(x, dims=[2])
    mv, _ = model(x_v)
    mv = torch.flip(mv, dims=[2])

    x_hv = torch.flip(x, dims=[2,3])
    mhv, _ = model(x_hv)
    mhv = torch.flip(mhv, dims=[2,3])

    return (m0 + mh + mv + mhv) / 4.0

def keep_largest_component(mask01: np.ndarray) -> np.ndarray:
    m = (mask01.astype(np.uint8) * 255)
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(m, connectivity=8)
    if num_labels <= 1:
        return mask01
    largest = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    out = (labels == largest).astype(np.uint8)
    return out

@torch.no_grad()
def dice_iou_postprocess(mask_logits, mask_gt, thr=0.5, eps=1e-7):
    probs = torch.sigmoid(mask_logits)
    pred = (probs > thr).float()

    preds_pp = []
    for b in range(pred.size(0)):
        m = pred[b,0].detach().cpu().numpy().astype(np.uint8)
        m = keep_largest_component(m)
        preds_pp.append(torch.from_numpy(m).to(pred.device).unsqueeze(0))
    pred_pp = torch.stack(preds_pp, dim=0).float()

    pred_flat = pred_pp.view(pred_pp.size(0), -1)
    gt_flat   = mask_gt.view(mask_gt.size(0), -1)

    inter = (pred_flat * gt_flat).sum(dim=1)
    dice = (2.0 * inter + eps) / (pred_flat.sum(dim=1) + gt_flat.sum(dim=1) + eps)
    iou  = (inter + eps) / (pred_flat.sum(dim=1) + gt_flat.sum(dim=1) - inter + eps)
    return dice.mean().item(), iou.mean().item()

In [ ]:
# ============================================================
# 10) Training / Validation loops (supports ablations)
# ============================================================
@dataclass
class ExperimentFlags:
    use_edge_head: bool = True
    use_patch_training: bool = True
    use_tta_eval: bool = True
    use_postprocess_eval: bool = True

@dataclass
class HyperParams:
    encoder: str = "timm-efficientnet-b7"
    lr: float = 3e-4
    epochs: int = 35
    patience: int = 8
    batch_size: int = 4
    num_workers: int = 2
    img_size_eval: int = 384
    patch_size: int = 256

def make_loaders(cfg: TrainConfig, hp: HyperParams):
    train_tfms, val_tfms = get_transforms(cfg)
    train_ds = PolypDatasetEdge(splits["train"], transform=train_tfms)
    val_ds   = PolypDatasetEdge(splits["val"], transform=val_tfms)
    test_ds  = PolypDatasetEdge(splits["test"], transform=val_tfms)

    train_loader = DataLoader(train_ds, batch_size=hp.batch_size, shuffle=True,
                              num_workers=hp.num_workers, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=hp.batch_size, shuffle=False,
                            num_workers=hp.num_workers, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=1, shuffle=False,
                             num_workers=hp.num_workers, pin_memory=True)
    external_loader = None
    if USE_EXTERNAL_TEST and len(external_pairs) > 0:
        ext_ds = PolypDatasetEdge(external_pairs, transform=val_tfms)
        external_loader = DataLoader(ext_ds, batch_size=1, shuffle=False,
                                     num_workers=hp.num_workers, pin_memory=True)
    return train_loader, val_loader, test_loader, external_loader

def run_one_epoch(model, loader, optimizer=None, flags: ExperimentFlags=None, scaler=None):
    train = optimizer is not None
    model.train() if train else model.eval()

    total_loss = 0.0
    total_seg  = 0.0
    total_edge = 0.0
    total_dice = 0.0
    total_iou  = 0.0
    n = 0

    for imgs, masks, edges in loader:
        imgs  = imgs.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)
        edges = edges.to(device, non_blocking=True)

        with torch.set_grad_enabled(train):
            with torch.cuda.amp.autocast(enabled=(device.type=="cuda")):
                mask_logits, edge_logits = model(imgs)

                loss_seg = seg_loss(mask_logits, masks)

                if flags.use_edge_head:
                    loss_edge = edge_loss(edge_logits, edges)
                    loss = 0.5 * loss_seg + 0.5 * loss_edge
                else:
                    loss_edge = torch.tensor(0.0, device=device)
                    loss = loss_seg

            if train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        d, j = dice_iou_from_logits(mask_logits.detach(), masks.detach())
        bs = imgs.size(0)
        total_loss += loss.item() * bs
        total_seg  += loss_seg.item() * bs
        total_edge += loss_edge.item() * bs
        total_dice += d * bs
        total_iou  += j * bs
        n += bs

    return (total_loss/n, total_seg/n, total_edge/n, total_dice/n, total_iou/n)

@torch.no_grad()
def evaluate(model, loader, flags: ExperimentFlags, name="Test"):
    model.eval()
    losses, dices, ious = [], [], []

    for imgs, masks, edges in loader:
        imgs = imgs.to(device)
        masks = masks.to(device)

        # inference
        if flags.use_tta_eval:
            mask_logits = predict_logits_tta(model, imgs)
        else:
            mask_logits, _ = model(imgs)

        # loss on seg only (for reporting)
        loss = seg_loss(mask_logits, masks).item()

        # metrics
        if flags.use_postprocess_eval:
            d, j = dice_iou_postprocess(mask_logits, masks)
        else:
            d, j = dice_iou_from_logits(mask_logits, masks)

        losses.append(loss); dices.append(d); ious.append(j)

    log(f"{name} -> Loss: {np.mean(losses):.4f} | Dice: {np.mean(dices):.4f} | IoU: {np.mean(ious):.4f}")
    return float(np.mean(losses)), float(np.mean(dices)), float(np.mean(ious))

def train_and_eval(flags: ExperimentFlags, hp: HyperParams, run_name: str):
    cfg = TrainConfig(use_patch_training=flags.use_patch_training,
                      patch_size=hp.patch_size,
                      img_size_eval=hp.img_size_eval)

    train_loader, val_loader, test_loader, external_loader = make_loaders(cfg, hp)

    model = UNetPPWithEdgeHead(encoder_name=hp.encoder).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=hp.lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=hp.epochs)
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

    best_val_dice = -1
    run_dir = OUT_DIR / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    
    best_path = run_dir / "best_model.pt"
    history_path = run_dir / "history.csv"
    metrics_path = run_dir / "metrics.json"
    pat = 0

    history = []

    for epoch in range(1, hp.epochs + 1):
        t0 = time.time()
        tr = run_one_epoch(model, train_loader, optimizer=optimizer, flags=flags, scaler=scaler)
        va = run_one_epoch(model, val_loader, optimizer=None, flags=flags, scaler=scaler)
        scheduler.step()

        tr_loss, tr_seg, tr_edge, tr_dice, tr_iou = tr
        va_loss, va_seg, va_edge, va_dice, va_iou = va

        improved = va_dice > best_val_dice
        if improved:
            best_val_dice = va_dice
            torch.save(model.state_dict(), best_path)
            pat = 0
        else:
            pat += 1

        dt = time.time() - t0
        log(f"Epoch {epoch:02d}/{hp.epochs} | "
              f"Train: loss {tr_loss:.4f}, seg {tr_seg:.4f}, edge {tr_edge:.4f}, dice {tr_dice:.4f} | "
              f"Val: loss {va_loss:.4f}, seg {va_seg:.4f}, edge {va_edge:.4f}, dice {va_dice:.4f} | "
              f"{'BEST' if improved else '':4s} | time {dt:.1f}s")

        history.append((epoch, tr_loss, va_loss, tr_dice, va_dice))

        if pat >= hp.patience:
            log("Early stopping triggered.")
            break

    log(f"Best val dice: {best_val_dice}")

    hist_df = pd.DataFrame(history, columns=["epoch", "train_loss", "val_loss", "train_dice", "val_dice"])
    hist_df.to_csv(history_path, index=False)
    log(f"Saved history: {history_path}")

    # Load best + evaluate
    model.load_state_dict(torch.load(best_path, map_location=device))
    k_loss, k_dice, k_iou = evaluate(model, test_loader, flags, name="Kvasir Test (Protocol A=100)")
    ext_dice = None
    if USE_EXTERNAL_TEST and external_loader is not None:
        _, ext_dice, _ = evaluate(model, external_loader, flags, name="External (CVC)")

    metrics_obj = {
        "run": run_name,
        "flags": {
            "use_edge_head": flags.use_edge_head,
            "use_patch_training": flags.use_patch_training,
            "use_tta_eval": flags.use_tta_eval,
            "use_postprocess_eval": flags.use_postprocess_eval,
        },
        "hp": hp.__dict__,
        "best_val_dice": float(best_val_dice),
        "kvasir_test": {"loss": float(k_loss), "dice": float(k_dice), "iou": float(k_iou)},
    }
    
    if USE_EXTERNAL_TEST and external_loader is not None:
        metrics_obj["external"] = {"dice": float(ext_dice)}
    
    with open(metrics_path, "w") as f:
        json.dump(metrics_obj, f, indent=2)
    
    log(f"Saved metrics: {metrics_path}")
    log(f"Saved checkpoint: {best_path}")

    return {
        "run": run_name,
        "edge_head": flags.use_edge_head,
        "patch_train": flags.use_patch_training,
        "tta": flags.use_tta_eval,
        "postprocess": flags.use_postprocess_eval,
        "best_val_dice": best_val_dice,
        "kvasir_test_dice": k_dice,
        "kvasir_test_iou": k_iou,
        "external_dice": ext_dice
    }

In [ ]:
# ============================================================
# 11) Ablation runner
# ============================================================
hp = HyperParams(
    encoder="timm-efficientnet-b7",
    lr=3e-4,
    epochs=35,
    patience=8,
    batch_size=4,
    num_workers=2,
    img_size_eval=384,
    patch_size=256
)

# Four ablation settings:
# A0: Baseline (no edge, no patch)  -> full image train, plain eval
# A1: +Edge head/loss (no patch)    -> full image train, plain eval
# A2: +Edge + Patch training        -> patch train, plain eval
# A3: +Edge + Patch + TTA+PP (final)-> patch train, strong eval

ablations = [
    ("A0_baseline", ExperimentFlags(use_edge_head=False, use_patch_training=False, use_tta_eval=False, use_postprocess_eval=False)),
    ("A1_edge",     ExperimentFlags(use_edge_head=True,  use_patch_training=False, use_tta_eval=False, use_postprocess_eval=False)),
    ("A2_edge_patch",ExperimentFlags(use_edge_head=True,  use_patch_training=True,  use_tta_eval=False, use_postprocess_eval=False)),
    ("A3_edge_patch_tta_pp", ExperimentFlags(use_edge_head=True, use_patch_training=True, use_tta_eval=True, use_postprocess_eval=True)),
]

# Set this to True if you want to run ALL ablations one after another.
# Otherwise set False and run a single experiment (recommended first).
RUN_ALL_ABLATIONS = True

results = []
if RUN_ALL_ABLATIONS:
    for name, flags in ablations:
        log("\n" + "="*60)
        log(f"Running: {name} | {flags}")
        log("="*60 + "\n")
        out = train_and_eval(flags, hp, run_name=name)
        results.append(out)
else:
    # Run only the final improved pipeline by default
    name, flags = ablations[-1]
    print("\nRunning single experiment:", name, flags, "\n")
    out = train_and_eval(flags, hp, run_name=name)
    results.append(out)

df = pd.DataFrame(results)
print("\n================ Ablation Results ================\n")
print(df)
print("\n=================================================\n")

# Save table for paper
ablation_csv = OUT_DIR / "ablation_results.csv"
df.to_csv(ablation_csv, index=False)
log(f"Saved ablation table: {ablation_csv}")

ablation_md = OUT_DIR / "ablation_results.md"
with open(ablation_md, "w") as f:
    f.write(df.to_markdown(index=False))
log(f"Saved ablation markdown: {ablation_md}")

In [ ]:
# ============================================================
# 12) Quick qualitative visualization on Kvasir test
# ============================================================
@torch.no_grad()
def show_samples(model, loader, flags, n=5, title="Samples"):
    model.eval()
    shown = 0
    plt.figure(figsize=(12, 3*n))
    for imgs, masks, edges in loader:
        imgs = imgs.to(device)
        masks = masks.to(device)

        if flags.use_tta_eval:
            mask_logits = predict_logits_tta(model, imgs)
        else:
            mask_logits, _ = model(imgs)

        probs = torch.sigmoid(mask_logits)
        pred = (probs > 0.5).float()

        img_np = imgs[0].detach().cpu().permute(1,2,0).numpy()
        img_np = np.clip(img_np, 0, 1)

        gt_np = masks[0,0].detach().cpu().numpy()
        pr_np = pred[0,0].detach().cpu().numpy().astype(np.uint8)
        if flags.use_postprocess_eval:
            pr_np = keep_largest_component(pr_np)

        r = shown*3
        plt.subplot(n, 3, r+1); plt.imshow(img_np); plt.title(f"{title} - Image"); plt.axis("off")
        plt.subplot(n, 3, r+2); plt.imshow(gt_np, cmap="gray"); plt.title("GT Mask"); plt.axis("off")
        plt.subplot(n, 3, r+3); plt.imshow(pr_np, cmap="gray"); plt.title("Pred Mask"); plt.axis("off")

        shown += 1
        if shown >= n:
            break
    plt.tight_layout()
    plt.show()

# If you ran single experiment, reload best model and show
# (reconstruct loaders for the chosen flags)
name, flags = (ablations[-1] if not RUN_ALL_ABLATIONS else ablations[-1])
cfg = TrainConfig(use_patch_training=flags.use_patch_training, patch_size=hp.patch_size, img_size_eval=hp.img_size_eval)
_, _, test_loader, _ = make_loaders(cfg, hp)

best_path = f"best_{name}.pt"
model = UNetPPWithEdgeHead(encoder_name=hp.encoder).to(device)
model.load_state_dict(torch.load(best_path, map_location=device))

SHOW_QUAL_SAMPLES = False
if SHOW_QUAL_SAMPLES:
    show_samples(model, test_loader, flags, n=5, title="Kvasir Test")